# ALBERT Fine-tuning for Multi-label Emotion Classification

Companion notebook to `roberta_3layer_head.ipynb` / `roberta_2layer_head.ipynb`. Same dataset,
preprocessing, and training loop, but fine-tunes **ALBERT** (`albert-base-v2`) instead of
RoBERTa, to compare a parameter-efficient transformer against RoBERTa on the same task
(SemEval 2025 Task 11-A, 5-label emotion classification: anger, fear, joy, sadness, surprise).

ALBERT underperformed RoBERTa on this task (mean F1 ~0.66-0.684 vs. RoBERTa's ~0.78), which this notebook reproduces the setup for.

In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AlbertModel, AlbertTokenizer, AdamW
from sklearn.metrics import f1_score, classification_report

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Specify the dataset path (adjust the path if needed)
dataset_path = '/content/drive/My Drive/SemEval'

In [ ]:
# Load the CSV file
def load_data(file_path, delimeter=","):
    df = pd.read_csv(file_path, delimiter=delimeter)
    texts = df["text"].tolist()
    labels = df[["anger", "fear", "joy", "sadness", "surprise"]].values
    return texts, labels

train_file = "/content/drive/My Drive/SemEval/track_a/train/eng.csv"
test_file = "/content/drive/My Drive/SemEval/track_a/dev/eng.csv"

train_texts, train_labels = load_data(train_file)
test_texts, test_labels = load_data(test_file)

# Initialize tokenizer
tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")

# Find the max token length dynamically
def find_max_length(texts, tokenizer):
    tokenized_texts = [tokenizer.tokenize(text) for text in texts]
    return max(len(tokens) for tokens in tokenized_texts)

max_length = find_max_length(train_texts + test_texts, tokenizer)
print(f"Dynamic max length: {max_length}")

def tokenize_texts(texts, tokenizer, max_length):
    return tokenizer(
        texts,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

train_encodings = tokenize_texts(train_texts, tokenizer, max_length)
test_encodings = tokenize_texts(test_texts, tokenizer, max_length)

**EmotionDataset**: same custom `Dataset` wrapper used in the RoBERTa notebooks, stores
tokenized inputs and multi-label targets as tensors.

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels=None, device="cpu"):
        self.encodings = {key: torch.tensor(val, dtype=torch.long).to(device) for key, val in encodings.items()}
        self.labels = torch.tensor(labels, dtype=torch.float).to(device) if labels is not None else None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

train_dataset = EmotionDataset(train_encodings, train_labels, device="cuda")
test_dataset = EmotionDataset(test_encodings, test_labels, device="cuda")

**AlbertClass**: `albert-base-v2` (hidden size 768) followed by the same three-layer
classification head shape used for RoBERTa (scaled down for ALBERT's smaller hidden size):
`fc1 (768 -> 512)`, `fc2 (512 -> 256)`, `fc3 (256 -> num_labels)`, each with LayerNorm, ReLU,
and Dropout.

In [ ]:
import torch.nn as nn
import random

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class AlbertClass(nn.Module):
    def __init__(self, num_labels=5):
        super(AlbertClass, self).__init__()
        self.albert = AlbertModel.from_pretrained("albert-base-v2")
        self.dropout = nn.Dropout(0.1)

        self.fc1 = nn.Linear(768, 512)
        self.layer_norm1 = nn.LayerNorm(512)
        self.fc2 = nn.Linear(512, 256)
        self.layer_norm2 = nn.LayerNorm(256)
        self.fc3 = nn.Linear(256, num_labels)

        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):
        output = self.albert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.pooler_output

        x = self.fc1(cls_output)
        x = self.layer_norm1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.layer_norm2(x)
        x = self.relu(x)
        x = self.dropout(x)

        logits = self.fc3(x)
        return logits

num_labels = 5
model = AlbertClass(num_labels=num_labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

In [ ]:
from transformers import get_scheduler
from tqdm import tqdm
import matplotlib.pyplot as plt

epochs = 20
num_training_steps = epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

loss_history = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linestyle='-', color='b')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('ALBERT Training Loss per Epoch')
plt.grid(True)
plt.show()

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.sigmoid(outputs).cpu().numpy()
            labels = labels.cpu().numpy()

            predictions.extend(preds)
            true_labels.extend(labels)

    predictions = np.array(predictions) > 0.5
    print(classification_report(true_labels, predictions, target_names=["anger", "fear", "joy", "sadness", "surprise"]))

evaluate(model, test_loader)

In [ ]:
model_save_path = "/content/drive/My Drive/SemEval/albert_emotion_model.pt"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")